# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/melikekaya01/flyrank-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Selected method

This notebook treats the task as a supervised binary classification and ranking problem. The target indicates whether a content item experienced a decline in recent search performance.

I start with Logistic Regression because it provides a simple and interpretable learned benchmark. Its coefficients help show the direction in which each feature contributes to the estimated probability of decline.

I also train a Random Forest as a nonlinear comparison model. It can capture interactions and nonlinear relationships that Logistic Regression may miss. However, the more complex model will only be preferred if it produces a meaningful improvement on the same validation split and metrics.

The model probabilities are used as ranking scores because the operational goal is to prioritize a limited number of content items for human review rather than automatically classify every page.

Both learned models will be compared with Melike's Week-4 baseline on the same held-out rows and the same ranking metrics. The primary metric is Precision@50, with Precision@20, average precision, ROC-AUC, and the target base rate reported as supporting metrics.

The results are interpreted as decision support. They do not establish that any feature causes performance decline or that reviewing a recommended item will necessarily improve future results.


In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Python version:", sys.version.split()[0])
print("Pandas version:", pd.__version__)
print("Random seed:", RANDOM_STATE)

Python version: 3.12.13
Pandas version: 2.2.2
Random seed: 42


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design

The train/test split is grouped by client using GroupShuffleSplit.

This design prevents content items from the same client from appearing in both the training and test sets. Without this separation, client-specific patterns could make the model appear more accurate than it would be for unseen clients.

Approximately 80% of the clients are used for training and 20% are held out for testing. Logistic Regression, Random Forest, and the Week-4 baseline will all be evaluated on exactly the same held-out rows and the same metrics.

The target and the columns used to generate it are excluded from the model features. Client and content identifiers are also excluded from training and are retained only for splitting and error inspection.


In [2]:
from pathlib import Path
import os

REPO_URL = "https://github.com/melikekaya01/flyrank-ml.git"
REPO_DIR = Path("/content/flyrank-ml")

# Remove a previously cloned different repository if necessary.
if REPO_DIR.exists():
    current_remote = os.popen(
        f"git -C {REPO_DIR} remote get-url origin 2>/dev/null"
    ).read().strip()

    if current_remote and "melikekaya01/flyrank-ml" not in current_remote:
        print("Removing previously cloned repository:", current_remote)
        !rm -rf /content/flyrank-ml

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Melike's repository already exists.")

os.chdir(REPO_DIR)

DATA_PATH = Path(
    "/content/flyrank-ml/data/raw/content_refresh_anonymized.csv"
)

print("Current directory:", Path.cwd())
print("Data file exists:", DATA_PATH.exists())

Cloning into '/content/flyrank-ml'...
remote: Enumerating objects: 139, done.
remote: Counting objects: 100% (139/139), done.
remote: Compressing objects: 100% (96/96), done.
remote: Total 139 (delta 48), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (139/139), 1.89 MiB | 26.17 MiB/s, done.
Resolving deltas: 100% (48/48), done.
Current directory: /content/flyrank-ml
Data file exists: True


In [3]:
# Load the dataset and create a leakage-safe grouped split

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset was not found at: {DATA_PATH}"
    )

df = pd.read_csv(DATA_PATH)

print("Dataset path:", DATA_PATH)
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")

required_columns = {
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
}

missing_columns = required_columns.difference(df.columns)

if missing_columns:
    raise ValueError(
        f"Required columns are missing: {sorted(missing_columns)}"
    )

# Proxy target: observed recent downward trend.
df["is_declining_label"] = (
    df["trend_direction"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("down")
    .astype(int)
)

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "provider_used",
    "model_used",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier",
]

feature_columns = numeric_features + categorical_features
target_column = "is_declining_label"
group_column = "client_id"

forbidden_features = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "content_id",
    "client_id",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
}

leaked_features = forbidden_features.intersection(feature_columns)

if leaked_features:
    raise ValueError(
        f"Leakage or identifier columns found: {sorted(leaked_features)}"
    )

model_df = df[
    [
        "content_id",
        group_column,
        target_column,
    ]
    + feature_columns
].copy()

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

train_idx, test_idx = next(
    splitter.split(
        model_df[feature_columns],
        model_df[target_column],
        groups=model_df[group_column],
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

X_train = train_df[feature_columns]
y_train = train_df[target_column]

X_test = test_df[feature_columns]
y_test = test_df[target_column]

train_clients = set(train_df[group_column])
test_clients = set(test_df[group_column])

print("\nSplit summary")
print("-" * 40)
print(f"Training rows: {len(train_df):,}")
print(f"Test rows: {len(test_df):,}")
print(f"Training clients: {len(train_clients):,}")
print(f"Test clients: {len(test_clients):,}")
print(
    "Client overlap:",
    len(train_clients.intersection(test_clients)),
)
print(f"Training decline rate: {y_train.mean():.2%}")
print(f"Test decline rate: {y_test.mean():.2%}")
print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")

assert train_clients.isdisjoint(test_clients)
assert target_column not in feature_columns
assert "trend_direction" not in feature_columns
assert "trend_pct" not in feature_columns
assert X_train.columns.tolist() == X_test.columns.tolist()

Dataset path: /content/flyrank-ml/data/raw/content_refresh_anonymized.csv
Rows: 30,000
Columns: 44

Split summary
----------------------------------------
Training rows: 23,837
Test rows: 6,163
Training clients: 25
Test clients: 7
Client overlap: 0
Training decline rate: 55.01%
Test decline rate: 51.10%
Numeric features: 22
Categorical features: 11


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### Training and baseline comparison

The preprocessing pipeline is fitted only on the training data. Numeric missing values are filled using training-set medians. Numeric variables are standardized for Logistic Regression, while categorical variables are filled with their most frequent training value and one-hot encoded.

Logistic Regression is used as the main interpretable learned model. Random Forest is included as a nonlinear comparison, but additional complexity will only be accepted if it improves the operational ranking metric.

Melike's Week-4 baseline is recreated using the original eligibility conditions and four scoring components: visibility, search position, low CTR, and low engagement.

All three methods are evaluated using the same held-out client groups, target, and metrics. The primary metric is Precision@50 because the practical objective is to create a small, high-quality human-review queue.


In [6]:
# Build preprocessing pipelines and train the learned models

numeric_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True,
            ),
        ),
    ]
)

logistic_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_preprocessor, numeric_features),
        ("categorical", categorical_preprocessor, categorical_features),
    ],
    remainder="drop",
)

forest_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                ]
            ),
            numeric_features,
        ),
        ("categorical", categorical_preprocessor, categorical_features),
    ],
    remainder="drop",
)

logistic_model = Pipeline(
    steps=[
        ("preprocessor", logistic_preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

random_forest_model = Pipeline(
    steps=[
        ("preprocessor", forest_preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=10,
                min_samples_leaf=10,
                max_features="sqrt",
                n_jobs=-1,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

print("Training Logistic Regression...")
logistic_model.fit(X_train, y_train)

print("Training Random Forest...")
random_forest_model.fit(X_train, y_train)

logistic_scores = logistic_model.predict_proba(X_test)[:, 1]
random_forest_scores = random_forest_model.predict_proba(X_test)[:, 1]

print("\nTraining complete")
print("-" * 40)
print(f"Test rows scored: {len(y_test):,}")
print(
    "Logistic score range:",
    f"{logistic_scores.min():.4f} to {logistic_scores.max():.4f}",
)
print(
    "Random Forest score range:",
    f"{random_forest_scores.min():.4f} to {random_forest_scores.max():.4f}",
)

Training Logistic Regression...
Training Random Forest...

Training complete
----------------------------------------
Test rows scored: 6,163
Logistic score range: 0.0324 to 0.9319
Random Forest score range: 0.0545 to 0.8921


In [7]:
# Recreate Melike's Week-4 CTR-opportunity baseline on the test set

baseline_test = test_df[
    [
        "content_id",
        "client_id",
        target_column,
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "avg_position",
        "engagement_rate",
    ]
].copy()

baseline_test["baseline_eligible"] = (
    (baseline_test["impressions_90d"] >= 500)
    & (baseline_test["avg_position"] <= 20)
    & (baseline_test["ctr"] < 0.5)
)

eligible_test = baseline_test.loc[
    baseline_test["baseline_eligible"]
].copy()

# Original Week-4 component formulas
eligible_test["visibility_score"] = (
    eligible_test["impressions_90d"]
    .rank(pct=True)
    .mul(100)
)

eligible_test["position_score"] = (
    (20 - eligible_test["avg_position"]) / 19 * 100
).clip(lower=0, upper=100)

eligible_test["low_ctr_score"] = (
    (0.5 - eligible_test["ctr"]) / 0.5 * 100
).clip(lower=0, upper=100)

eligible_test["low_engagement_score"] = (
    100
    - eligible_test["engagement_rate"]
    .rank(pct=True)
    .mul(100)
)

eligible_test["action_score"] = (
    0.40 * eligible_test["visibility_score"]
    + 0.30 * eligible_test["position_score"]
    + 0.20 * eligible_test["low_ctr_score"]
    + 0.10 * eligible_test["low_engagement_score"]
)

baseline_test = baseline_test.merge(
    eligible_test[
        [
            "content_id",
            "visibility_score",
            "position_score",
            "low_ctr_score",
            "low_engagement_score",
            "action_score",
        ]
    ],
    on="content_id",
    how="left",
)

# Eligible rows rank above ineligible rows.
# The original action score orders rows within the eligible queue.
baseline_test["baseline_score"] = np.where(
    baseline_test["baseline_eligible"],
    1000 + baseline_test["action_score"].fillna(0),
    0,
)

baseline_scores = baseline_test["baseline_score"].to_numpy()

print("Baseline test summary")
print("-" * 40)
print(
    "Eligible test rows:",
    f"{baseline_test['baseline_eligible'].sum():,}",
)
print(
    "Eligible test share:",
    f"{baseline_test['baseline_eligible'].mean():.2%}",
)
print(
    "Maximum eligible action score:",
    f"{eligible_test['action_score'].max():.4f}",
)
print(
    "Minimum eligible action score:",
    f"{eligible_test['action_score'].min():.4f}",
)

assert len(baseline_scores) == len(y_test)
assert baseline_test.loc[
    baseline_test["baseline_eligible"],
    "action_score",
].notna().all()

Baseline test summary
----------------------------------------
Eligible test rows: 1,827
Eligible test share: 29.64%
Maximum eligible action score: 90.8798
Minimum eligible action score: 6.9749


In [8]:
# Compare all methods on the same held-out clients

def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))
    top_indices = np.argsort(-scores, kind="stable")[:k]

    return float(y_true[top_indices].mean())


def recall_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    total_positives = y_true.sum()

    if total_positives == 0:
        return np.nan

    k = min(k, len(y_true))
    top_indices = np.argsort(-scores, kind="stable")[:k]

    return float(
        y_true[top_indices].sum() / total_positives
    )


def evaluate_method(method_name, y_true, scores):
    return {
        "method": method_name,
        "precision_at_20": precision_at_k(
            y_true,
            scores,
            20,
        ),
        "precision_at_50": precision_at_k(
            y_true,
            scores,
            50,
        ),
        "recall_at_50": recall_at_k(
            y_true,
            scores,
            50,
        ),
        "average_precision": average_precision_score(
            y_true,
            scores,
        ),
        "roc_auc": roc_auc_score(
            y_true,
            scores,
        ),
    }


comparison_table = pd.DataFrame(
    [
        evaluate_method(
            "Week-4 CTR baseline",
            y_test,
            baseline_scores,
        ),
        evaluate_method(
            "Logistic Regression",
            y_test,
            logistic_scores,
        ),
        evaluate_method(
            "Random Forest",
            y_test,
            random_forest_scores,
        ),
    ]
)

comparison_table["test_base_rate"] = y_test.mean()

metric_columns = [
    "precision_at_20",
    "precision_at_50",
    "recall_at_50",
    "average_precision",
    "roc_auc",
    "test_base_rate",
]

comparison_table[metric_columns] = comparison_table[
    metric_columns
].round(4)

comparison_table = comparison_table.sort_values(
    by="precision_at_50",
    ascending=False,
).reset_index(drop=True)

display(comparison_table)

best_method = comparison_table.loc[0, "method"]
best_precision_50 = comparison_table.loc[
    0,
    "precision_at_50",
]

baseline_precision_50 = comparison_table.loc[
    comparison_table["method"].eq(
        "Week-4 CTR baseline"
    ),
    "precision_at_50",
].iloc[0]

print(
    f"Best method by Precision@50: {best_method}"
)
print(
    f"Best Precision@50: {best_precision_50:.2%}"
)
print(
    f"Baseline Precision@50: "
    f"{baseline_precision_50:.2%}"
)
print(
    "Absolute improvement over baseline:",
    f"{best_precision_50 - baseline_precision_50:+.2%}",
)

,method,precision_at_20,precision_at_50,recall_at_50,average_precision,roc_auc,test_base_rate
0,Week-4 CTR baseline,0.75,0.68,0.0108,0.5223,0.5202,0.511
1,Logistic Regression,0.70,0.64,0.0102,0.5691,0.5777,0.511
2,Random Forest,0.60,0.58,0.0092,0.5898,0.6111,0.511


Best method by Precision@50: Week-4 CTR baseline
Best Precision@50: 68.00%
Baseline Precision@50: 68.00%
Absolute improvement over baseline: +0.00%


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [9]:
# Inspect top-ranked correct and incorrect recommendations

evaluation_df = test_df[
    [
        "content_id",
        "client_id",
        target_column,
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "avg_position",
        "engagement_rate",
        "days_since_last_update",
    ]
].copy()

evaluation_df["baseline_score"] = baseline_scores
evaluation_df["logistic_score"] = logistic_scores
evaluation_df["random_forest_score"] = random_forest_scores


def top_k_error_summary(data, score_column, method_name, k=50):
    ranked = (
        data.sort_values(score_column, ascending=False)
        .head(k)
        .copy()
    )

    ranked["prediction_result"] = np.where(
        ranked[target_column].eq(1),
        "true_positive",
        "false_positive",
    )

    summary = pd.DataFrame(
        {
            "method": [method_name],
            "top_k": [k],
            "true_positives": [ranked[target_column].sum()],
            "false_positives": [(ranked[target_column] == 0).sum()],
            "precision_at_k": [ranked[target_column].mean()],
            "median_impressions": [ranked["impressions_90d"].median()],
            "median_ctr": [ranked["ctr"].median()],
            "median_position": [ranked["avg_position"].median()],
            "median_engagement_rate": [
                ranked["engagement_rate"].median()
            ],
            "median_days_since_update": [
                ranked["days_since_last_update"].median()
            ],
        }
    )

    return ranked, summary


baseline_top50, baseline_summary = top_k_error_summary(
    evaluation_df,
    "baseline_score",
    "Week-4 CTR baseline",
)

logistic_top50, logistic_summary = top_k_error_summary(
    evaluation_df,
    "logistic_score",
    "Logistic Regression",
)

forest_top50, forest_summary = top_k_error_summary(
    evaluation_df,
    "random_forest_score",
    "Random Forest",
)

error_summary_table = pd.concat(
    [
        baseline_summary,
        logistic_summary,
        forest_summary,
    ],
    ignore_index=True,
)

display(error_summary_table.round(4))

,method,top_k,true_positives,false_positives,precision_at_k,median_impressions,median_ctr,median_position,median_engagement_rate,median_days_since_update
0,Week-4 CTR baseline,50,34,16,0.68,33762.5,0.065,4.90,0.0,20.0
1,Logistic Regression,50,32,18,0.64,1151.0,0.065,7.70,0.0,102.0
2,Random Forest,50,29,21,0.58,1181.5,0.070,16.85,0.0,104.0


In [10]:
# Display the highest-ranked false positives for each method

display_columns = [
    "content_id",
    "client_id",
    target_column,
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "days_since_last_update",
]

print("Week-4 CTR baseline false positives")
display(
    baseline_top50.loc[
        baseline_top50[target_column].eq(0),
        display_columns + ["baseline_score"],
    ]
    .sort_values("baseline_score", ascending=False)
    .head(10)
)

print("\nLogistic Regression false positives")
display(
    logistic_top50.loc[
        logistic_top50[target_column].eq(0),
        display_columns + ["logistic_score"],
    ]
    .sort_values("logistic_score", ascending=False)
    .head(10)
)

print("\nRandom Forest false positives")
display(
    forest_top50.loc[
        forest_top50[target_column].eq(0),
        display_columns + ["random_forest_score"],
    ]
    .sort_values("random_forest_score", ascending=False)
    .head(10)
)

Week-4 CTR baseline false positives


,content_id,client_id,is_declining_label,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate,days_since_last_update,baseline_score
16736,content_e12868d1f396,client_4e07408562,0,149712,104,0.07,2.9,5.94,7,1086.578216
22928,content_929aa622b6a0,client_4e07408562,0,11301,3,0.03,2.4,0.00,104,1086.272013
11163,content_ceaa28bba4ca,client_4e07408562,0,43287,26,0.06,3.9,2.17,104,1085.136543
22028,content_73c54f78c06a,client_f369cb89fc,0,213963,211,0.10,4.7,0.86,20,1083.994786
12869,content_5d5653c4eb4f,client_4e07408562,0,15101,0,0.00,5.7,0.00,7,1083.750266
28461,content_e420ff479117,client_4e07408562,0,21024,10,0.05,5.5,0.00,104,1083.532942
26384,content_396019fec61a,client_4e07408562,0,83362,74,0.09,3.8,7.81,104,1083.432149
8138,content_d02df0d34dd5,client_4e07408562,0,39163,48,0.12,5.5,0.00,14,1083.031792
28511,content_0b5de68b61a2,client_f369cb89fc,0,33187,34,0.10,5.8,0.00,20,1082.876444
13631,content_d274ac4158ef,client_4e07408562,0,65138,6,0.01,6.8,4.00,26,1082.779270



Logistic Regression false positives


,content_id,client_id,is_declining_label,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate,days_since_last_update,logistic_score
20736,content_41baf0722ad9,client_8527a891e2,0,3115,0,0.00,12.8,0.00,104,0.911621
10175,content_374e795aab68,client_f369cb89fc,0,235,2,0.85,31.0,0.00,20,0.906780
11887,content_ce59581533ca,client_8527a891e2,0,289,2,0.69,18.8,0.00,102,0.898231
18531,content_d10f9ce1e0cd,client_4e07408562,0,166,1,0.60,16.1,0.00,104,0.897735
26614,content_7be5f150dc65,client_f369cb89fc,0,290,0,0.00,5.9,0.00,20,0.897301
4905,content_f0d98be4b42c,client_4e07408562,0,5818,9,0.15,5.1,22.22,104,0.896786
4050,content_500bd3907331,client_4e07408562,0,4037,4,0.10,5.5,0.00,104,0.888608
12332,content_4d9f36001f06,client_8527a891e2,0,3369,1,0.03,13.2,33.33,104,0.888305
6739,content_f45787e64ac2,client_4e07408562,0,291,1,0.34,5.2,0.00,104,0.885422
11061,content_0b47dae0c7f9,client_8527a891e2,0,1191,0,0.00,23.1,0.00,103,0.880565



Random Forest false positives


,content_id,client_id,is_declining_label,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate,days_since_last_update,random_forest_score
22042,content_2ba626fea4d6,client_8527a891e2,0,360,0,0.00,7.2,0.0,104,0.882247
22526,content_1d0963b56227,client_4e07408562,0,3445,3,0.09,39.0,20.0,104,0.881700
10080,content_35d63627bf3e,client_8527a891e2,0,1525,0,0.00,32.6,0.0,103,0.879676
11061,content_0b47dae0c7f9,client_8527a891e2,0,1191,0,0.00,23.1,0.0,103,0.879419
4050,content_500bd3907331,client_4e07408562,0,4037,4,0.10,5.5,0.0,104,0.876053
12069,content_ff4370afd49c,client_4e07408562,0,1677,3,0.18,33.1,0.0,104,0.872719
22524,content_846bb4dd8b44,client_8527a891e2,0,870,1,0.11,17.6,0.0,104,0.872672
5399,content_6677fd6c4ea5,client_4e07408562,0,1152,2,0.17,34.8,0.0,104,0.869715
29456,content_b46c62b14582,client_8527a891e2,0,6240,8,0.13,31.8,0.0,103,0.865132
20736,content_41baf0722ad9,client_8527a891e2,0,3115,0,0.00,12.8,0.0,104,0.865020


In [11]:
# Measure held-out feature importance for Random Forest

permutation_result = permutation_importance(
    random_forest_model,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

permutation_table = pd.DataFrame(
    {
        "feature": feature_columns,
        "importance_mean": permutation_result.importances_mean,
        "importance_std": permutation_result.importances_std,
    }
).sort_values(
    "importance_mean",
    ascending=False,
)

print("Top Random Forest permutation importances")
display(permutation_table.head(15).round(4))

Top Random Forest permutation importances


,feature,importance_mean,importance_std
13,days_with_impressions,0.0360,0.0042
5,impressions_90d,0.0131,0.0019
18,avg_position,0.0070,0.0012
6,clicks_90d,0.0062,0.0005
17,ctr,0.0048,0.0002
31,impression_tier,0.0029,0.0005
32,position_tier,0.0028,0.0010
20,scroll_rate,0.0025,0.0008
14,days_with_sessions,0.0025,0.0003
15,content_age_days,0.0023,0.0016


In [12]:
# Inspect the strongest Logistic Regression coefficients

logistic_preprocessor_fitted = (
    logistic_model.named_steps["preprocessor"]
)

logistic_classifier = (
    logistic_model.named_steps["classifier"]
)

transformed_feature_names = (
    logistic_preprocessor_fitted.get_feature_names_out()
)

coefficient_table = pd.DataFrame(
    {
        "feature": transformed_feature_names,
        "coefficient": logistic_classifier.coef_[0],
    }
)

coefficient_table["absolute_coefficient"] = (
    coefficient_table["coefficient"].abs()
)

print("Features associated with higher estimated decline probability")
display(
    coefficient_table
    .sort_values("coefficient", ascending=False)
    .head(10)
    .round(4)
)

print("Features associated with lower estimated decline probability")
display(
    coefficient_table
    .sort_values("coefficient", ascending=True)
    .head(10)
    .round(4)
)

Features associated with higher estimated decline probability


,feature,coefficient,absolute_coefficient
8,numeric__sessions_90d,0.8594,0.8594
13,numeric__days_with_impressions,0.7808,0.7808
37,categorical__model_used_gpt-5-mini,0.5912,0.5912
27,categorical__content_type_keyword article,0.5246,0.5246
43,categorical__freshness_tier_0-30,0.4361,0.4361
3,numeric__word_count,0.3892,0.3892
47,categorical__word_count_tier_1000-2000,0.3837,0.3837
62,categorical__position_tier_striking,0.3181,0.3181
40,categorical__age_tier_31-90,0.3011,0.3011
61,categorical__position_tier_page_3_5,0.2712,0.2712


Features associated with lower estimated decline probability


,feature,coefficient,absolute_coefficient
63,categorical__position_tier_top_3,-1.0304,1.0304
9,numeric__users_90d,-0.9694,0.9694
26,categorical__content_type_feedly article,-0.5441,0.5441
38,categorical__model_used_unknown,-0.5085,0.5085
14,numeric__days_with_sessions,-0.5024,0.5024
30,categorical__main_intent_navigational,-0.4511,0.4511
45,categorical__freshness_tier_31-90,-0.4458,0.4458
15,numeric__content_age_days,-0.3902,0.3902
44,categorical__freshness_tier_181+,-0.3662,0.3662
58,categorical__impression_tier_moderate,-0.3609,0.3609


### Model comparison

The Week-4 CTR-opportunity baseline achieved the strongest operational ranking result. Its Precision@50 was 0.68, meaning that 34 of the first 50 recommended content items belonged to the observed decline class.

Logistic Regression achieved a Precision@50 of 0.64, corresponding to 32 decline examples in its first 50 recommendations. Random Forest achieved a Precision@50 of 0.58, corresponding to 29 decline examples.

The learned models therefore did not improve the primary operational metric. The simpler Week-4 rule should not be replaced merely because the alternatives are more complex.

Random Forest achieved the highest ROC-AUC at 0.6111, followed by Logistic Regression at 0.5777 and the Week-4 baseline at 0.5202. Random Forest also achieved the highest average precision at 0.5898. This suggests that the forest captured some useful signal across the complete test set, even though it did not create the strongest top-50 queue.

The difference between these results is important. ROC-AUC evaluates broad discrimination across many possible thresholds, while Precision@50 evaluates the small review queue that is most relevant to the operational decision.

### Error interpretation

The Week-4 baseline produced 16 false positives in its first 50 recommendations. These content items satisfied the CTR-opportunity criteria because they combined meaningful visibility, relatively favorable search position, low CTR, and low engagement, but they did not belong to the observed decline class.

This shows that a content item can appear to offer a CTR opportunity without experiencing a future performance decline. Low CTR or low engagement may reflect search intent, SERP layout, advertisements, branded-query composition, content purpose, or query-level behavior that is not available in the aggregated dataset.

Logistic Regression produced 18 false positives in its first 50 recommendations, while Random Forest produced 21. The learned models therefore made more top-ranked errors than the focused business rule.

The target base rate in the held-out clients was 0.511. All methods performed above this rate at Precision@50, but the Week-4 baseline created the largest practical lift for the limited review queue.

### Final method decision

The Week-4 CTR-opportunity baseline remains the recommended method for prioritizing content items because it achieved the best Precision@50 on unseen client groups.

Random Forest showed stronger broad discrimination and may still be useful for future experimentation. A sensible next step would be a hybrid design in which the original Week-4 rule creates an eligible candidate set and a learned model reranks only those eligible items.

These results are predictive and associational. They do not demonstrate that CTR, engagement, search position, content age, or another observed feature causes future decline.


### Self-check

* The train/test split is grouped by client, and no client appears in both sets.
* The target and target-generating trend columns are excluded from the feature set.
* Identifiers are used only for grouping and error inspection.
* Logistic Regression, Random Forest, and the Week-4 baseline are evaluated on the same test rows.
* The comparison uses the same target and metrics for every method.
* Precision@50 is the primary metric because the output is a limited human-review queue.
* The notebook reports supporting metrics, including Precision@20, recall at 50, average precision, ROC-AUC, and the test base rate.
* The more complex models are not selected because they did not improve the primary metric.
* Errors and feature contributions are inspected on held-out clients.
* The conclusions are framed as decision support rather than causal claims.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.